In [ ]:
!pip install openai langchain langchain-openai langgraph crewai requests python-dotenv -q

In [ ]:
# ============================================================
# IA_agente_Camanchaca4.ipynb
# IL2.4 - Arquitectura y Buenas Prácticas
# Proyecto: Monitoreo Climático - Salmones Camanchaca
# Python 3.11.9
# ============================================================

import os
import re
import json
import time
import requests
from datetime import datetime
from typing import Dict, List, Any, Optional
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langgraph.prebuilt import create_react_agent

load_dotenv()

if not os.getenv("OPENAI_BASE_URL"):
    raise ValueError("Falta OPENAI_BASE_URL en .env")
if not os.getenv("GITHUB_TOKEN"):
    raise ValueError("Falta GITHUB_TOKEN en .env")

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0
)

print("✓ Modelo configurado con streaming habilitado")
print(f"Modelo: {llm.model_name}")
print(f"Streaming: {llm.streaming}")


In [ ]:
# ============================================================
# SECCIÓN 2: DOCUMENTACIÓN DE ARQUITECTURA
# Ref: 1-architecture_example.py del repositorio de la materia
# Descripción formal del sistema agente Camanchaca
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║     ARQUITECTURA DEL SISTEMA AGENTE - SALMONES CAMANCHACA    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  OBJETIVO DEL SISTEMA:                                       ║
║  Monitorear condiciones climáticas en tiempo real de los     ║
║  centros de cultivo (Ensenada, Puelche, Huito) y apoyar      ║
║  decisiones operativas del equipo de producción.             ║
║                                                              ║
║  CAPA 1 - PRESENTACIÓN                                       ║
║     └─ Jupyter Notebooks como interfaz para operadores       ║
║                                                              ║
║  CAPA 2 - APLICACIÓN (Agentes)                               ║
║     ├─ Agente ReAct (LangGraph)  → Razonamiento paso a paso  ║
║     ├─ Memoria de Sesión         → Continuidad conversacional ║
║     └─ Crew Multi-Agente (CrewAI) → Planificación compleja   ║
║                                                              ║
║  CAPA 3 - DOMINIO (Herramientas / Tools)                     ║
║     ├─ get_clima_actual      → Condiciones en tiempo real    ║
║     ├─ get_pronostico_semana → Pronóstico 7 días             ║
║     ├─ evaluar_operacion     → Aptitud para operar           ║
║     └─ get_mejor_dia_operacion → Día óptimo por actividad    ║
║                                                              ║
║  CAPA 4 - INFRAESTRUCTURA (Datos)                            ║
║     └─ Open-Meteo API (sin API key, gratuita)                ║
║        · Ensenada: Lat -41.14, Lon -72.40                    ║
║        · Puelche:  Lat -41.73, Lon -73.60                    ║
║        · Huito:    Lat -41.78, Lon -73.58                    ║
║                                                              ║
║  FLUJO PRINCIPAL:                                            ║
║  Operador → Agente → Herramienta → Open-Meteo API            ║
║           ↓                                                  ║
║      Memoria de Sesión (contexto entre turnos)               ║
║           ↓                                                  ║
║  Respuesta técnica con recomendación operativa               ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


In [ ]:
# ============================================================
# SECCIÓN 3: CONFIGURACIÓN CENTRALIZADA
# Buena práctica: todos los parámetros en un único lugar
# Ref: 2-best_practices.py del repositorio de la materia
# ============================================================

CENTROS = {
    "ensenada": {"lat": -41.140459, "lon": -72.404236, "nombre": "Piscicultura Petrohué"},
    "puelche":  {"lat": -41.733,    "lon": -73.602,    "nombre": "Centro Puelche"},
    "huito":    {"lat": -41.783,    "lon": -73.583,    "nombre": "Centro Huito (San José)"}
}

LIMITES_OPERATIVOS = {
    "viento_max_kmh": 40,
    "lluvia_max_mm":  10,
    "temp_min_c":      5,
    "temp_max_c":     18,
    "temp_optima_min": 8,
    "temp_optima_max": 14
}

OPERACIONES_VALIDAS = ["cosecha", "biometría", "tratamiento", "alimentación", "mantenimiento"]

print("✓ Configuración centralizada cargada.")
print(f"  Centros registrados:  {list(CENTROS.keys())}")
print(f"  Operaciones válidas:  {OPERACIONES_VALIDAS}")
print(f"  Límites operativos:   {LIMITES_OPERATIVOS}")


In [ ]:
# ============================================================
# SECCIÓN 4: FUNCIONES DE VALIDACIÓN Y UTILIDAD
# Principio DRY: centralizar lógica reutilizable
# Buena práctica: separación de responsabilidades
# ============================================================

def validar_centro(centro: str) -> tuple:
    """
    Valida que el centro exista en el sistema.

    Args:
        centro: Nombre del centro a validar.

    Returns:
        Tupla (es_valido: bool, mensaje: str)
    """
    if not centro or not isinstance(centro, str):
        return False, "El parámetro 'centro' no puede estar vacío."
    if centro.lower() not in CENTROS:
        return False, (
            f"Centro '{centro}' no encontrado. "
            f"Centros disponibles: {', '.join(CENTROS.keys())}."
        )
    return True, "ok"


def validar_operacion(operacion: str) -> tuple:
    """
    Valida que la operación sea reconocida por el sistema.

    Args:
        operacion: Tipo de operación a validar.

    Returns:
        Tupla (es_valido: bool, mensaje: str)
    """
    if not operacion or not isinstance(operacion, str):
        return False, "El parámetro 'operacion' no puede estar vacío."
    if operacion.lower() not in OPERACIONES_VALIDAS:
        return False, (
            f"Operación '{operacion}' no reconocida. "
            f"Operaciones válidas: {', '.join(OPERACIONES_VALIDAS)}."
        )
    return True, "ok"


def obtener_datos_climaticos(lat: float, lon: float, modo: str = "actual") -> dict:
    """
    Función centralizada para obtener datos de Open-Meteo API.
    Aplica el principio DRY (Don't Repeat Yourself).

    Args:
        lat:  Latitud del centro.
        lon:  Longitud del centro.
        modo: 'actual' para clima en tiempo real, 'pronostico' para 7 días.

    Returns:
        Diccionario con los datos climáticos o error.
    """
    try:
        if modo == "actual":
            url = (
                f"https://api.open-meteo.com/v1/forecast"
                f"?latitude={lat}&longitude={lon}"
                f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
                f"&timezone=America/Santiago"
            )
        else:
            url = (
                f"https://api.open-meteo.com/v1/forecast"
                f"?latitude={lat}&longitude={lon}"
                f"&daily=temperature_2m_max,temperature_2m_min,"
                f"precipitation_sum,wind_speed_10m_max,weathercode"
                f"&timezone=America/Santiago"
            )

        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return {"ok": True, "data": response.json()}

    except requests.exceptions.Timeout:
        return {"ok": False, "error": "Timeout: La API de Open-Meteo no respondió."}
    except requests.exceptions.ConnectionError:
        return {"ok": False, "error": "Sin conexión: Verifica tu acceso a internet."}
    except requests.exceptions.HTTPError as e:
        return {"ok": False, "error": f"Error HTTP: {e}"}
    except Exception as e:
        return {"ok": False, "error": f"Error inesperado: {e}"}


print("✓ Funciones de validación y utilidad definidas.")


In [ ]:
# ============================================================
# SECCIÓN 5: HERRAMIENTAS CON BUENAS PRÁCTICAS APLICADAS
# - Docstrings completos (Args, Returns)
# - Validación de parámetros antes de llamar la API
# - Manejo de errores robusto
# - Uso de función centralizada obtener_datos_climaticos()
# ============================================================

@tool
def get_clima_actual(centro: str) -> str:
    """
    Obtiene el clima actual para un centro de cultivo de Camanchaca.

    Consulta la API Open-Meteo con las coordenadas del centro seleccionado
    y retorna temperatura, viento, precipitación y condición general.

    Args:
        centro: Nombre del centro. Opciones: ensenada, puelche, huito.

    Returns:
        String con las condiciones climáticas actuales del centro.
    """
    es_valido, mensaje = validar_centro(centro)
    if not es_valido:
        return mensaje

    datos    = CENTROS[centro.lower()]
    resultado = obtener_datos_climaticos(datos["lat"], datos["lon"], modo="actual")

    if not resultado["ok"]:
        return f"Error al consultar clima en {datos['nombre']}: {resultado['error']}"

    current   = resultado["data"]["current"]
    temp      = current["temperature_2m"]
    viento    = current["wind_speed_10m"]
    lluvia    = current["precipitation"]
    codigo    = current["weathercode"]
    condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"

    return (
        f"Centro: {datos['nombre']}\n"
        f"Temperatura: {temp}°C\n"
        f"Viento: {viento} km/h\n"
        f"Precipitación: {lluvia} mm\n"
        f"Condición: {condicion}"
    )


@tool
def get_pronostico_semana(centro: str) -> str:
    """
    Obtiene el pronóstico climático de 7 días para un centro de cultivo.

    Consulta la API Open-Meteo y entrega temperaturas máximas y mínimas,
    precipitación acumulada, viento máximo y condición general por día.

    Args:
        centro: Nombre del centro. Opciones: ensenada, puelche, huito.

    Returns:
        String con el pronóstico día a día para los próximos 7 días.
    """
    es_valido, mensaje = validar_centro(centro)
    if not es_valido:
        return mensaje

    datos    = CENTROS[centro.lower()]
    resultado = obtener_datos_climaticos(datos["lat"], datos["lon"], modo="pronostico")

    if not resultado["ok"]:
        return f"Error al consultar pronóstico en {datos['nombre']}: {resultado['error']}"

    daily   = resultado["data"]["daily"]
    resumen = f"Pronóstico 7 días - {datos['nombre']}:\n"

    for i in range(7):
        fecha     = daily["time"][i]
        tmax      = daily["temperature_2m_max"][i]
        tmin      = daily["temperature_2m_min"][i]
        lluvia    = daily["precipitation_sum"][i]
        viento    = daily["wind_speed_10m_max"][i]
        codigo    = daily["weathercode"][i]
        condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"

        resumen += (
            f"\n{fecha}: {tmin}°C - {tmax}°C | "
            f"Viento: {viento} km/h | "
            f"Lluvia: {lluvia} mm | {condicion}"
        )
    return resumen


@tool
def evaluar_operacion(centro: str, operacion: str) -> str:
    """
    Evalúa si las condiciones climáticas actuales son seguras para
    realizar una operación específica en un centro de cultivo.

    Compara los datos en tiempo real con los límites operativos definidos
    por los protocolos de seguridad de Salmones Camanchaca.

    Args:
        centro:    Nombre del centro. Opciones: ensenada, puelche, huito.
        operacion: Tipo de operación. Opciones: cosecha, biometría,
                   tratamiento, alimentación, mantenimiento.

    Returns:
        String con evaluación de aptitud y alertas si corresponde.
    """
    valido_centro, msg_centro = validar_centro(centro)
    if not valido_centro:
        return msg_centro

    valido_op, msg_op = validar_operacion(operacion)
    if not valido_op:
        return msg_op

    datos    = CENTROS[centro.lower()]
    resultado = obtener_datos_climaticos(datos["lat"], datos["lon"], modo="actual")

    if not resultado["ok"]:
        return f"Error al consultar clima: {resultado['error']}"

    current = resultado["data"]["current"]
    viento  = current["wind_speed_10m"]
    lluvia  = current["precipitation"]
    temp    = current["temperature_2m"]

    alertas = []
    if viento > LIMITES_OPERATIVOS["viento_max_kmh"]:
        alertas.append(
            f"⚠️ Viento peligroso: {viento} km/h "
            f"(límite: {LIMITES_OPERATIVOS['viento_max_kmh']} km/h)"
        )
    if lluvia > LIMITES_OPERATIVOS["lluvia_max_mm"]:
        alertas.append(f"⚠️ Lluvia intensa: {lluvia} mm")
    if temp < LIMITES_OPERATIVOS["temp_min_c"]:
        alertas.append(f"⚠️ Temperatura muy baja: {temp}°C")
    if temp > LIMITES_OPERATIVOS["temp_max_c"]:
        alertas.append(f"⚠️ Temperatura elevada: {temp}°C (riesgo para FCR)")

    if not alertas:
        return (
            f"✅ Condiciones APTAS para {operacion} en {datos['nombre']}.\n"
            f"Temperatura: {temp}°C | Viento: {viento} km/h | Lluvia: {lluvia} mm"
        )
    return (
        f"❌ Condiciones NO APTAS para {operacion} en {datos['nombre']}:\n"
        + "\n".join(alertas)
    )


@tool
def get_mejor_dia_operacion(centro: str, operacion: str) -> str:
    """
    Determina el mejor día de la semana para realizar una operación
    en un centro de cultivo, basándose en el pronóstico climático.

    Calcula un puntaje de aptitud (0-100) para cada día considerando
    viento, lluvia y temperatura según los límites operativos.

    Args:
        centro:    Nombre del centro. Opciones: ensenada, puelche, huito.
        operacion: Tipo de operación a planificar.

    Returns:
        String con el mejor día recomendado y sus condiciones esperadas.
    """
    valido_centro, msg_centro = validar_centro(centro)
    if not valido_centro:
        return msg_centro

    datos    = CENTROS[centro.lower()]
    resultado = obtener_datos_climaticos(datos["lat"], datos["lon"], modo="pronostico")

    if not resultado["ok"]:
        return f"Error al consultar pronóstico: {resultado['error']}"

    daily       = resultado["data"]["daily"]
    mejor_dia   = None
    mejor_score = -1

    for i in range(7):
        fecha  = daily["time"][i]
        tmax   = daily["temperature_2m_max"][i]
        tmin   = daily["temperature_2m_min"][i]
        lluvia = daily["precipitation_sum"][i]
        viento = daily["wind_speed_10m_max"][i]

        score = 100
        if viento > LIMITES_OPERATIVOS["viento_max_kmh"]:
            score -= 50
        elif viento > 25:
            score -= 20

        if lluvia > LIMITES_OPERATIVOS["lluvia_max_mm"]:
            score -= 40
        elif lluvia > 5:
            score -= 15

        if tmax > LIMITES_OPERATIVOS["temp_max_c"]:
            score -= 20
        elif tmin < LIMITES_OPERATIVOS["temp_min_c"]:
            score -= 10

        if score > mejor_score:
            mejor_score = score
            mejor_dia   = {
                "fecha": fecha, "score": score,
                "tmin": tmin,   "tmax": tmax,
                "lluvia": lluvia, "viento": viento
            }

    return (
        f"Mejor día para {operacion} en {datos['nombre']}:\n"
        f"Fecha:       {mejor_dia['fecha']}\n"
        f"Temperatura: {mejor_dia['tmin']}°C - {mejor_dia['tmax']}°C\n"
        f"Viento máx:  {mejor_dia['viento']} km/h\n"
        f"Lluvia acum: {mejor_dia['lluvia']} mm\n"
        f"Puntaje:     {mejor_dia['score']}/100"
    )


tools = [get_clima_actual, get_pronostico_semana, evaluar_operacion, get_mejor_dia_operacion]

print("✓ Herramientas con buenas prácticas definidas.")
print(f"  Total herramientas: {len(tools)}")
print(f"  Herramientas: {[t.name for t in tools]}")


In [ ]:
# ============================================================
# SECCIÓN 6: SISTEMA AGENTE COMPLETO E INTEGRADO
# Integra memoria, streaming y todas las herramientas
# Patrón AgenteOrquestador: ref 1-architecture_example.py
# ============================================================

history_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """
    Recupera o crea el historial de mensajes para una sesión.

    Args:
        session_id: Identificador único de la sesión operativa.

    Returns:
        Instancia de InMemoryChatMessageHistory para la sesión.
    """
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]


agent_executor = create_react_agent(llm, tools)


def invocar_sistema(consulta: str, session_id: str) -> str:
    """
    Invoca el agente manteniendo el historial de la sesión.

    Args:
        consulta:   Texto de la consulta del operador.
        session_id: ID de sesión para mantener contexto.

    Returns:
        Respuesta del agente como string.
    """
    hist = get_session_history(session_id)
    messages = [
        {"role": "user" if isinstance(m, HumanMessage) else "assistant",
         "content": m.content}
        for m in hist.messages
    ] + [{"role": "user", "content": consulta}]

    response = agent_executor.invoke({"messages": messages})
    output   = response["messages"][-1].content

    hist.add_user_message(consulta)
    hist.add_ai_message(output)
    return output


print("✓ Sistema agente Camanchaca completamente integrado.")
print(f"  Herramientas registradas: {[t.name for t in tools]}")
print(f"  Centros activos: {list(CENTROS.keys())}")


In [ ]:
# ============================================================
# SECCIÓN 7: PRUEBAS DEL SISTEMA
# Validación de cada componente antes del despliegue
# Buena práctica: siempre probar antes de producción
# ============================================================

print("=== PRUEBAS DEL SISTEMA AGENTE CAMANCHACA ===\n")

errores = []

print("1. Probando validación de centros:")
casos_centros = [
    ("ensenada",    True),
    ("PUELCHE",     True),
    ("huito",       True),
    ("magallanes",  False),
    ("",            False),
]
for centro, esperado in casos_centros:
    valido, _ = validar_centro(centro)
    estado     = "✅" if valido == esperado else "❌"
    if valido != esperado:
        errores.append(f"validar_centro('{centro}')")
    print(f"   {estado} validar_centro('{centro}') → {valido}")

print("\n2. Probando validación de operaciones:")
casos_ops = [
    ("cosecha",     True),
    ("biometría",   True),
    ("tratamiento", True),
    ("vuelo",       False),
    ("",            False),
]
for op, esperado in casos_ops:
    valido, _ = validar_operacion(op)
    estado     = "✅" if valido == esperado else "❌"
    if valido != esperado:
        errores.append(f"validar_operacion('{op}')")
    print(f"   {estado} validar_operacion('{op}') → {valido}")

print("\n3. Probando conexión con Open-Meteo API:")
resultado_api = obtener_datos_climaticos(-41.140459, -72.404236, modo="actual")
if resultado_api["ok"]:
    print("   ✅ Conexión exitosa con Open-Meteo API.")
else:
    print(f"   ❌ Error de conexión: {resultado_api['error']}")
    errores.append("conexion_api")

print("\n4. Probando herramienta get_clima_actual:")
try:
    clima = get_clima_actual.invoke("ensenada")
    print("   ✅ get_clima_actual funciona correctamente.")
    print(f"   Muestra: {clima[:60]}...")
except Exception as e:
    print(f"   ❌ Error: {e}")
    errores.append("get_clima_actual")

print("\n5. Probando herramienta evaluar_operacion:")
try:
    eval_op = evaluar_operacion.invoke({"centro": "puelche", "operacion": "cosecha"})
    print("   ✅ evaluar_operacion funciona correctamente.")
    print(f"   Muestra: {eval_op[:60]}...")
except Exception as e:
    print(f"   ❌ Error: {e}")
    errores.append("evaluar_operacion")

print(f"\n{'='*50}")
if not errores:
    print("✅ TODAS LAS PRUEBAS PASARON — Sistema listo para producción.")
else:
    print(f"❌ {len(errores)} prueba(s) fallaron: {errores}")


In [ ]:
# ============================================================
# SECCIÓN 8: DEMOSTRACIÓN DEL SISTEMA COMPLETO
# Simula una jornada operativa real con memoria y streaming
# ============================================================

print("=== DEMOSTRACIÓN SISTEMA COMPLETO CAMANCHACA ===\n")

session_id = f"turno_{datetime.now().strftime('%Y%m%d_%H%M')}"

jornada_operativa = [
    "Buenos días, soy el jefe del turno de mañana. ¿Cómo están las condiciones en los tres centros hoy?",
    "¿Es seguro proceder con la cosecha en Ensenada esta mañana?",
    "¿Cuál es el mejor día esta semana para hacer biometría en Puelche?",
    "Resúmeme qué operaciones son viables hoy y cuáles deberíamos postergar."
]

for i, consulta in enumerate(jornada_operativa, 1):
    print(f"\n{'='*55}")
    print(f"👤 Operador (Turno {session_id}) — Consulta {i}:")
    print(f"   {consulta}")
    print(f"{'='*55}")
    print("🤖 Agente Camanchaca:")

    try:
        respuesta = invocar_sistema(consulta, session_id)
        print(f"\n📋 Respuesta: {respuesta}\n")
        time.sleep(3)
    except Exception as e:
        print(f"❌ Error en consulta {i}: {e}")

print(f"\n{'='*55}")
print("📊 RESUMEN DE LA SESIÓN:")
if session_id in history_store:
    historial = history_store[session_id].messages
    print(f"   Session ID:       {session_id}")
    print(f"   Total mensajes:   {len(historial)}")
    print(f"   Consultas:        {len(jornada_operativa)}")
    print(f"   Herramientas:     {[t.name for t in tools]}")
    print(f"   Centros activos:  {list(CENTROS.keys())}")
else:
    print("   Sin mensajes registrados en esta sesión.")


In [ ]:
# ============================================================
# SECCIÓN 9: RESUMEN DE BUENAS PRÁCTICAS APLICADAS
# Ref: 2-best_practices.py del repositorio de la materia
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║        BUENAS PRÁCTICAS APLICADAS EN ESTE PROYECTO          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. SEPARACIÓN DE RESPONSABILIDADES                          ║
║     ├─ Validación centralizada en validar_centro()           ║
║     ├─ Acceso a API centralizado en obtener_datos()          ║
║     └─ Lógica de negocio en cada herramienta (@tool)         ║
║                                                              ║
║  2. MANEJO DE ERRORES ROBUSTO                                ║
║     ├─ Timeout de conexión capturado explícitamente          ║
║     ├─ Errores HTTP manejados con raise_for_status()         ║
║     └─ Validación de parámetros antes de llamar la API       ║
║                                                              ║
║  3. PRINCIPIO DRY (Don't Repeat Yourself)                    ║
║     └─ obtener_datos_climaticos() centraliza las llamadas    ║
║        a Open-Meteo, evitando código duplicado               ║
║                                                              ║
║  4. DOCUMENTACIÓN CON DOCSTRINGS                             ║
║     └─ Cada función documenta Args, Returns y propósito      ║
║                                                              ║
║  5. CONFIGURACIÓN CENTRALIZADA                               ║
║     ├─ CENTROS: coordenadas en un solo lugar                 ║
║     ├─ LIMITES_OPERATIVOS: umbrales en un solo lugar         ║
║     └─ OPERACIONES_VALIDAS: lista de operaciones permitidas  ║
║                                                              ║
║  6. MEMORIA Y CONTEXTO                                       ║
║     └─ InMemoryChatMessageHistory mantiene el contexto       ║
║        de la sesión operativa entre consultas                ║
║                                                              ║
║  7. STREAMING                                                ║
║     └─ StreamingStdOutCallbackHandler mejora la experiencia  ║
║        del operador al ver la respuesta en tiempo real       ║
║                                                              ║
║  8. PRUEBAS DEL SISTEMA                                      ║
║     └─ Validación de cada componente antes de ejecutar       ║
║        el sistema completo en producción                     ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


In [ ]:
# ============================================================
# SECCIÓN 10: FLUJO DE TRABAJO AUTOMATIZADO
# Documentación del flujo completo del sistema
# ============================================================

print("=== FLUJO DE TRABAJO AUTOMATIZADO — SALMONES CAMANCHACA ===\n")

flujo = [
    ("Paso 1", "Operador",        "Consulta en lenguaje natural",                "Input al agente"),
    ("Paso 2", "Agente LangGraph", "Razona qué herramienta usar (ReAct)",         "Tool call generado"),
    ("Paso 3", "Herramienta",     "Consulta Open-Meteo API",                      "Datos climáticos JSON"),
    ("Paso 4", "Herramienta",     "Valida parámetros y evalúa límites operativos","Evaluación de aptitud"),
    ("Paso 5", "Agente LangGraph", "Sintetiza respuesta con contexto",             "Recomendación operativa"),
    ("Paso 6", "Memoria",         "Guarda interacción en session history",         "Contexto para próxima consulta"),
]

for paso, actor, accion, resultado in flujo:
    print(f"  [{paso}] {actor}")
    print(f"    Acción:    {accion}")
    print(f"    Resultado: {resultado}\n")

print("=== COMPARACIÓN DE COMPONENTES ===\n")

componentes = {
    "LangGraph (create_react_agent)": {
        "Rol":         "Motor de razonamiento principal",
        "Ventaja":     "Nativo en LangChain 1.3+, sin AgentExecutor",
        "Uso":         "Ciclo ReAct: Razonar → Actuar → Observar"
    },
    "Open-Meteo API": {
        "Rol":         "Fuente de datos climáticos externos",
        "Ventaja":     "Gratuita, sin API key, cobertura Los Lagos",
        "Uso":         "Temperatura, viento, lluvia en tiempo real"
    },
    "InMemoryChatMessageHistory": {
        "Rol":         "Memoria conversacional por sesión",
        "Ventaja":     "Nativa en langchain_core, sin dependencias extra",
        "Uso":         "Mantiene contexto entre turnos del operador"
    },
    "StreamingStdOutCallbackHandler": {
        "Rol":         "Visualización en tiempo real",
        "Ventaja":     "Mejora percepción de velocidad del sistema",
        "Uso":         "El operador ve la respuesta generarse token a token"
    }
}

for componente, info in componentes.items():
    print(f"📌 {componente}:")
    for clave, valor in info.items():
        print(f"   {clave}: {valor}")
    print()
